In [ ]:
!pip install -q openai requests pydantic python-dotenv


# AI Agents Seminar - Complete Implementation

**Topics:** Hugging Face Agents, Smol Agents, LlamaIndex, LangGraph

На этом семинаре мы изучим различные подходы к созданию AI агентов и их практической реализации.

## Секция 1: Введение в AI агентов (10 минут)

### Что такое AI агенты?

AI агент — это система, которая может:
1. **Воспринимать** окружение через входные данные
2. **Рассуждать** о том, какие действия предпринять
3. **Действовать** используя инструменты или генерируя выходные данные

### Ключевые компоненты:
- **LLM (Large Language Model)**: Мозг для рассуждений
- **Инструменты (Tools)**: Внешние функции, которые агент может вызывать
- **Память (Memory)**: Контекст и история разговора
- **Планирование (Planning)**: Стратегия для многошаговых задач

### Агент vs Традиционный AI:
- **Традиционный AI**: Вход → Модель → Выход
- **Agent AI**: Вход → Рассуждение → Использование инструментов → Еще рассуждение → Выход

### Hugging Face Transformers Agents

В этом семинаре мы будем использовать подходы, основанные на курсе **Hugging Face Agents**:
- **ReAct Pattern**: Паттерн Reasoning + Acting, который является основой для большинства современных агентов
- **Tool Calling**: Механизм вызова инструментов через LLM
- **Iterative Reasoning**: Итеративное рассуждение с наблюдением результатов

Hugging Face Transformers Agents предоставляет готовые инструменты для работы с моделями, но мы реализуем базовый паттерн вручную для лучшего понимания.

In [ ]:
# Импорты и настройка
import os
import json
from datetime import datetime
from typing import Any, Dict, List
from openai import OpenAI

# Configuration
# Важно: Укажите свой OpenRouter API ключ здесь или через переменную окружения
# Получить ключ можно на https://openrouter.ai/keys
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")

if OPENROUTER_API_KEY == "":
    print("Внимание: Укажите свой OpenRouter API ключ!")
    print("1. Получите ключ на https://openrouter.ai/keys")
    print("2. Замените 'YOUR_API_KEY_HERE' выше или установите переменную окружения:")
    print("   export OPENROUTER_API_KEY='your-key-here'")
else:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
    print("OpenRouter API ключ установлен")

# Initialize OpenRouter client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "AI Agents Seminar"
    }
)

print("OpenRouter client initialized successfully")

# Используем дешёвую модель с поддержкой tool calling
# Qwen 2.5 72B
DEFAULT_MODEL = "qwen/qwen-2.5-72b-instruct"

print(f"\nИспользуемая модель: {DEFAULT_MODEL}")
print("Эта модель поддерживает tool calling и относительно дешевая")
print("\nПроверить цены и доступность моделей:")
print(" https://openrouter.ai/models")


Внимание: Укажите свой OpenRouter API ключ!
1. Получите ключ на https://openrouter.ai/keys
2. Замените 'YOUR_API_KEY_HERE' выше или установите переменную окружения:
   export OPENROUTER_API_KEY='your-key-here'

Или вставьте ключ прямо в код (не рекомендуется для продакшена):
OPENROUTER_API_KEY = 'sk-or-v1-...'
OpenRouter client initialized successfully

Используемая модель: qwen/qwen-2.5-72b-instruct
Эта модель поддерживает tool calling и относительно дешевая

Стоимость: GPT-3.5 Turbo ~$0.0015 за 1K токенов
 (Для семинара на 1 час обычно нужно ~$0.10-0.50)

Проверить цены и доступность моделей:
 https://openrouter.ai/models


In [ ]:
# Определение инструментов для агента

def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")

def calculate(operation: str, a: float, b: float) -> float:
    """Perform basic mathematical operations."""
    operations = {
        "+": a + b,
        "-": a - b,
        "*": a * b,
        "/": a / b if b != 0 else "Error: Division by zero"
    }
    return operations.get(operation, "Error: Invalid operation")

def search_knowledge_base(query: str) -> str:
    """Search a mock knowledge base for information."""
    knowledge_base = {
        "agent": "An AI agent is a system that can perceive, reason, and act autonomously",
        "react": "ReAct (Reasoning + Acting) is a pattern that combines reasoning and acting in language models. It follows a loop: Thought -> Action -> Observation -> repeat",
        "react pattern": "ReAct (Reasoning + Acting) is a pattern that combines reasoning and acting in language models. It follows a loop: Thought -> Action -> Observation -> repeat. This pattern is the foundation of Hugging Face Transformers Agents.",
        "tool calling": "Tool calling allows LLMs to execute external functions",
        "hugging face agents": "Hugging Face Transformers Agents provides ready-to-use tools for building agents. The HfAgent class implements the ReAct pattern for iterative reasoning and tool use.",
        "langchain": "LangChain is a framework for developing LLM applications",
        "llamaindex": "LlamaIndex is a framework for building RAG applications",
        "smol agents": "Smol Agents is a lightweight framework focusing on simplicity and efficiency"
    }
    # Try exact match first, then partial match
    query_lower = query.lower()
    if query_lower in knowledge_base:
        return knowledge_base[query_lower]
    # Try partial matches
    for key, value in knowledge_base.items():
        if key in query_lower or query_lower in key:
            return value
    return f"No information found for '{query}'"

# Map function names to actual functions
function_map = {
"get_current_time": get_current_time,
"calculate": calculate,
"search_knowledge_base": search_knowledge_base
}

# Tool definitions for the LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Получить текущие дату и время",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Выполнить математические вычисления",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["+", "-", "*", "/"]
                    },
                    "a": {"type": "number"},
                    "b": {"type": "number"}
                },
                "required": ["operation", "a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Поиск информации в базе знаний",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    }
]

print(f"Defined {len(function_map)} agent tools")

Defined 3 agent tools


### Реализация паттерна ReAct

**ReAct (Reasoning + Acting)** — это паттерн, который объединяет рассуждение и действие в итеративном цикле. Этот паттерн является основой курса Hugging Face Agents.

Цикл ReAct:
1. **Thought (Мысль)**: LLM рассуждает о том, что делать дальше
2. **Action (Действие)**: Выбор и выполнение инструмента
3. **Observation (Наблюдение)**: Обработка результата инструмента
4. **Повторение** до завершения задачи

Этот паттерн позволяет агенту:
- Планировать многошаговые задачи
- Адаптироваться к результатам предыдущих действий
- Использовать внешние инструменты для расширения возможностей LLM

В Hugging Face Transformers Agents этот паттерн реализован через класс `HfAgent`, но мы создадим свою реализацию для понимания внутренней работы.

In [ ]:
import time
from openai import APIError

def run_react_agent(user_message: str, model: str = DEFAULT_MODEL, max_iterations: int = 5, max_retries: int = 3):
    """
    Run an agent using the ReAct pattern with error handling and retry logic.

    Based on Hugging Face Agents course - ReAct (Reasoning + Acting) pattern:
    1. Thought: Reason about what to do next
    2. Action: Choose and execute a tool
    3. Observation: Process the tool's result
    4. Repeat until task is complete
    """
    messages = [{"role": "user", "content": user_message}]

    print(f"Starting ReAct agent with model: {model}")
    print(f"User request: {user_message}")
    print("-" * 50)

    for iteration in range(max_iterations):
        print(f"\nIteration {iteration + 1}:")
        response = None
        for retry in range(max_retries):
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=tools,
                    tool_choice="auto",
                    temperature=0.1,
                    max_tokens=1500
                )
                break
            except APIError as e:
                if retry < max_retries - 1:
                    wait_time = 2 ** retry
                    print(f"API error (attempt {retry + 1}/{max_retries}): {e}")
                    print(f"Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    print(f"Failed after {max_retries} attempts: {e}")
                    return f"Agent failed: {e}"
            except Exception as e:
                print(f"Unexpected error: {e}")
                return f"Agent error: {e}"

        if response is None:
            return "Agent failed to get response"

        choice = response.choices[0]
        print(f"Status: {choice.finish_reason}")

        if choice.finish_reason == "stop":
            final_answer = choice.message.content
            print(f"\nFinal Answer: {final_answer}")
            return final_answer

        elif choice.finish_reason == "tool_calls":
            assistant_message = choice.message
            messages.append({
                "role": "assistant",
                "content": assistant_message.content or "",
                "tool_calls": assistant_message.tool_calls
            })

            for tool_call in assistant_message.tool_calls:
                func_name = tool_call.function.name
                # Handle arguments - can be None, dict, or JSON string
                args_raw = tool_call.function.arguments
                if args_raw is None:
                    func_args = {}
                elif isinstance(args_raw, dict):
                    func_args = args_raw
                else:
                    func_args = json.loads(args_raw)

                print(f"Action: {func_name}({func_args})")

                result = function_map[func_name](**func_args)
                print(f"Observation: {result}")

                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": func_name,
                    "content": str(result)
                })

    return "Agent did not complete the task within the iteration limit"


In [ ]:
# Тестирование ReAct агента
print("\n" + "="*70)
print("Тестирование ReAct агента")
print("="*70)

result = run_react_agent(
"What time is it now? Also, please calculate 15 * 8 and tell me about ReAct pattern."
)


Тестирование ReAct агента
Starting ReAct agent with model: qwen/qwen-2.5-72b-instruct
User request: What time is it now? Also, please calculate 15 * 8 and tell me about ReAct pattern.
--------------------------------------------------

Iteration 1:
Status: tool_calls
Action: get_current_time({})
Observation: 2025-11-13 16:23:07 UTC
Action: calculate({'a': 15, 'b': 8, 'operation': '*'})
Observation: 120
Action: search_knowledge_base({'query': 'ReAct pattern'})
Observation: ReAct (Reasoning + Acting) is a pattern that combines reasoning and acting in language models. It follows a loop: Thought -> Action -> Observation -> repeat. This pattern is the foundation of Hugging Face Transformers Agents.

Iteration 2:
API error (attempt 1/3): Error code: 422 - {'error': {'message': 'Provider returned error', 'code': 422, 'metadata': {'raw': '{"detail":[{"type":"literal_error","loc":["body","messages",1,"ChatCompletionToolMessage","role"],"msg":"Input should be <ChatMessageRole.TOOL: \'tool\'>","

---

## Задание 1: Добавление нового инструмента к ReAct агенту

**Цель** Научиться расширять функциональность агента, добавляя новые инструменты.

**Что нужно сделать**
1. Создайте новый инструмент для агента (например, конвертер валют, генератор случайных чисел, или что-то другое)
2. Добавьте его в `function_map` и `tools`
3. Протестируйте агента с новым инструментом

**Подсказки**
- Используйте существующие инструменты как пример
- Не забудьте добавить описание параметров в формат JSON Schema
- Протестируйте с запросом, который требует использования нового инструмента

**Примеры запросов:**
- "Сгенерируй случайное число от 1 до 100 и умножь его на 5"
- "Конвертируй 100 долларов в евро" (если делаете конвертер валют)

In [ ]:
import random

In [ ]:
# Задание 1: Добавьте новый инструмент здесь

# Шаг 1: Создайте функцию для нового инструмента
def generate_random(min_val: int, max_val: int) -> int:
    """
    Generate random number in range min/max val
    """
    return random.randint(min_val, max_val)

# Шаг 2: Добавьте функцию в function_map
function_map["generate_random"] = generate_random

# Шаг 3: Добавьте описание инструмента в tools
tools.append({
    "type": "function",
    "function": {
        "name": "generate_random",
        "description": "Generate random number in range min/max val",
        "parameters": {
            "type": "object",
            "properties": {
                "min_val": {"type": "integer", "description": "minimum value"},
                "max_val": {"type": "integer", "description": "maximum value"}
            },
            "required": ["min_val", "max_val"]
        }
    }
})

# Шаг 4: Протестируйте агента
result = run_react_agent("Сгенирируй мне случайное число между 2 и 22")
print(result)

Starting ReAct agent with model: qwen/qwen-2.5-72b-instruct
User request: Сгенирируй мне случайное число между 2 и 22
--------------------------------------------------

Iteration 1:
Status: tool_calls
Action: generate_random({'min_val': 2, 'max_val': 22})
Observation: 6

Iteration 2:
Status: stop

Final Answer: Случайное число между 2 и 22, которое было сгенерировано, это 6.
Случайное число между 2 и 22, которое было сгенерировано, это 6.


## Секция 2: Smol Agents-like arch

**Smol Agents** фокусируется на:
- Минимальных зависимостях
- Простом для понимания коде
- Быстром выполнении
- Простой интеграции инструментов

In [ ]:
class SmolAgent:
    """A minimal agent implementation focusing on simplicity and efficiency."""

    def __init__(self, client, tools, function_map):
        self.client = client
        self.tools = tools
        self.function_map = function_map
        self.conversation_history = []

    def run(self, user_input: str, model: str = DEFAULT_MODEL, max_retries: int = 3) -> str:
        """Run the agent with minimal overhead."""
        self.conversation_history.append({"role": "user", "content": user_input})

        # Retry logic for API errors
        for retry in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=model,
                    messages=self.conversation_history,
                    tools=self.tools,
                    tool_choice="auto",
                    temperature=0.1
                )
                break
            except APIError as e:
                if retry < max_retries - 1:
                    wait_time = 2 ** retry
                    print(f"API error (attempt {retry + 1}/{max_retries}): {e}")
                    print(f"Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    return f"Agent failed: {e}"
            except Exception as e:
                return f"Agent error: {e}"

        choice = response.choices[0]

        if choice.finish_reason == "tool_calls":
            assistant_message = choice.message
            self.conversation_history.append({
                "role": "assistant",
                "content": assistant_message.content or "",
                "tool_calls": assistant_message.tool_calls
            })

            for tool_call in assistant_message.tool_calls:
                func_name = tool_call.function.name
                # Handle arguments - can be None, dict, or JSON string
                args_raw = tool_call.function.arguments
                if args_raw is None:
                    func_args = {}
                elif isinstance(args_raw, dict):
                    func_args = args_raw
                else:
                    func_args = json.loads(args_raw)
                result = self.function_map[func_name](**func_args)

                self.conversation_history.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": func_name,
                    "content": str(result)
                })

            # Get final response after tool execution
            final_response = self.client.chat.completions.create(
                model=model,
                messages=self.conversation_history,
                temperature=0.1
            )

            return final_response.choices[0].message.content

        return choice.message.content

# Create and test smol agent
smol_agent = SmolAgent(client, tools, function_map)
print("Smol Agent created")

# Test Smol Agent
print("\nTesting Smol Agent:")
smol_result = smol_agent.run("Calculate 25 + 17 and tell me the current time")
print(f"Smol Agent Result: {smol_result}")

Smol Agent created

Testing Smol Agent:
Smol Agent Result: The calculation of 25 + 17 is indeed 42. The current time is 16:36:28 UTC on November 13, 2025.


---

## Задание 2

**Цель** Научиться кастомизировать агента, добавляя новые возможности.

**Что нужно сделать**
1. Добавьте счетчик количества использованных инструментов в класс `ImprovedSmolAgent`
2. Модифицируйте метод `run`, чтобы он подсчитывал и выводил количество использованных инструментов

**Подсказки**
- В методе `__init__` добавьте атрибут `self.tool_call_count = 0` для хранения счетчика
- В методе `run`, в цикле обработки `tool_calls`, увеличивайте счетчик: `self.tool_call_count += 1`
- В конце метода `run` выведите статистику: `print(f"Использовано инструментов: {self.tool_call_count}")`

**Примечание:** В ячейке ниже уже есть готовая структура кода. Вам нужно только убедиться, что счетчик правильно увеличивается и выводится статистика.

In [ ]:
# Задание 2: Добавьте счетчик инструментов в SmolAgent

class ImprovedSmolAgent(SmolAgent):
    """Улучшенная версия SmolAgent со счетчиком инструментов."""

    def __init__(self, client, tools, function_map):
        # Инициализируем базовый класс
        super().__init__(client, tools, function_map)
        # TO DO добавить счётчик инструментов
        self.tool_call_count = 0


    def run(self, user_input: str, model: str = DEFAULT_MODEL, max_retries: int = 3) -> str:
        """Run с подсчетом использованных инструментов."""
        # Добавляем запрос пользователя в историю
        self.conversation_history.append({"role": "user", "content": user_input})

        # Retry logic for API errors
        response = None
        for retry in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=model,
                    messages=self.conversation_history,
                    tools=self.tools,
                    tool_choice="auto",
                    temperature=0.1
                )
                break
            except APIError as e:
                if retry < max_retries - 1:
                    wait_time = 2 ** retry
                    print(f"API error (attempt {retry + 1}/{max_retries}): {e}")
                    print(f"Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    return f"Agent failed: {e}"
            except Exception as e:
                return f"Agent error: {e}"

        if response is None:
            return "Agent failed to get response"

        choice = response.choices[0]

        if choice.finish_reason == "tool_calls":
            assistant_message = choice.message
            self.conversation_history.append({
                "role": "assistant",
                "content": assistant_message.content or "",
                "tool_calls": assistant_message.tool_calls
            })

            # Обрабатываем каждый вызов инструмента
            for tool_call in assistant_message.tool_calls:
                func_name = tool_call.function.name
                # Увеличиваем счетчик при каждом вызове инструмента
                self.tool_call_count += 1

                # Handle arguments
                args_raw = tool_call.function.arguments
                if args_raw is None:
                    func_args = {}
                elif isinstance(args_raw, dict):
                    func_args = args_raw
                else:
                    func_args = json.loads(args_raw)

                result = self.function_map[func_name](**func_args)

                self.conversation_history.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": func_name,
                    "content": str(result)
                })

            # Get final response after tool execution
            final_response = self.client.chat.completions.create(
                model=model,
                messages=self.conversation_history,
                temperature=0.1
            )

            response_text = final_response.choices[0].message.content
        else:
            response_text = choice.message.content

        # Выводим статистику использования инструментов
        print(f"Использовано инструментов: {self.tool_call_count}")

        return response_text

# Протестируйте улучшенный агент
print("=" * 70)
print("Тестирование улучшенного Smol Agent")
print("=" * 70)
improved_agent = ImprovedSmolAgent(client, tools, function_map)
result = improved_agent.run("Calculate 10 * 5 and tell me the current time")
print(f"\nРезультат: {result}")

Тестирование улучшенного Smol Agent
Использовано инструментов: 1

Результат: The calculation of 10 * 5 is indeed 50. As for the current time, it's currently **14:30** (UTC). 

Please note that the time provided is based on my current context and may not reflect the exact time when you read this. If you need the exact time, you can check your device's clock.


### RAG Agent

In [ ]:
class SimpleRAGAgent:
    """A simple RAG agent for demonstration purposes."""

    def __init__(self, client):
        self.client = client
        # Mock document store
        self.documents = {
            "agents": "AI agents are autonomous systems that can perceive their environment, make decisions, and take actions to achieve specific goals. They combine large language models with tools and memory.",
            "rag": "Retrieval-Augmented Generation (RAG) is a technique that enhances language models by retrieving relevant information from external knowledge sources before generating responses.",
            "llamaindex": "LlamaIndex is a framework that connects LLMs with external data sources, enabling the creation of knowledge-augmented applications through indexing and retrieval.",
            "langchain": "LangChain is a framework for developing applications powered by language models, providing tools for chaining LLM calls and integrating with external data sources.",
            "react_pattern": "The ReAct pattern combines reasoning and acting in iterative loops, allowing agents to think about problems step by step while taking actions and observing results."
        }

    def retrieve_documents(self, query: str, top_k: int = 2) -> List[str]:
        """Simple keyword-based document retrieval."""
        query_lower = query.lower()
        relevant_docs = []

        for key, doc in self.documents.items():
            if any(word in doc.lower() for word in query_lower.split()):
                relevant_docs.append(doc)

        return relevant_docs[:top_k]

    def query(self, user_question: str, model: str = DEFAULT_MODEL) -> str:
        """Query the RAG agent with context retrieval."""
        # Retrieve relevant documents
        relevant_docs = self.retrieve_documents(user_question)

        # Create context-enhanced prompt
        context = "\n\n".join(relevant_docs)
        enhanced_prompt = f"""Context information:
{context}

Question: {user_question}

Please answer the question using the provided context information. If the context doesn't contain relevant information, say so clearly."""

        response = self.client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": enhanced_prompt}],
            temperature=0.1,
            max_tokens=1000
        )

        return response.choices[0].message.content

# Create and test RAG agent
rag_agent = SimpleRAGAgent(client)
print("RAG Agent created")

# Test RAG Agent
print("\nTesting RAG Agent:")
rag_result = rag_agent.query("What is the ReAct pattern and how does it work?")
print(f"RAG Agent Result: {rag_result}")

RAG Agent created

Testing RAG Agent:
RAG Agent Result: The provided context information does not contain specific details about the ReAct pattern. However, based on general knowledge, the ReAct (Reasoning and Acting) pattern is a framework designed to improve the decision-making and problem-solving capabilities of AI agents. It works by enabling AI agents to reason about their environment and the tasks they need to accomplish, and then act accordingly. The ReAct pattern typically involves the following steps:

1. **Perception**: The AI agent perceives its environment, gathering information through sensors or other means.
2. **Reasoning**: The agent processes the perceived information, using reasoning to understand the current state of the environment and to plan the next steps.
3. **Action**: Based on the reasoning, the agent decides on and executes an action to achieve its goals.
4. **Feedback**: The agent receives feedback from the environment, which it uses to refine its reasoning 

---

## Задание 3: Расширение RAG агента

**Цель** Научиться работать с базой знаний и улучшать поиск информации.

**Что нужно сделать**
1. Расширьте базу документов `self.documents` в `SimpleRAGAgent`, добавив минимум 3 новых документа на интересующую вас тему
2. Улучшите метод `retrieve_documents`, чтобы он учитывал не только ключевые слова, но и синонимы
3. Добавьте параметр `top_k` в метод `query`, чтобы можно было настраивать количество возвращаемых документов

**Подсказки**
- Используйте словарь синонимов для улучшения поиска
- Попробуйте простой алгоритм подсчета совпадений для ранжирования документов
- Протестируйте на вопросах, которые требуют информации из нескольких документов

**Дополнительно** Реализуйте простой алгоритм ранжирования документов по релевантности.

In [ ]:
class EnhancedRAGAgent(SimpleRAGAgent):
    """Улучшенная версия RAG агента с синонимами и ранжированием."""

    def __init__(self, client):
        super().__init__(client)

        # Расширяем базу документов новыми темами
        self.documents.update({
            "python": "Python is a high-level programming language known for its simplicity and readability. It's widely used in data science, web development, and AI applications.",
            "machine_learning": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
            "neural_networks": "Neural networks are computing systems inspired by biological neural networks. They consist of interconnected nodes (neurons) that process information.",
            "deep_learning": "Deep learning is a subset of machine learning that uses neural networks with multiple layers to learn complex patterns in data.",
            "nlp": "Natural Language Processing (NLP) is a field of AI that focuses on the interaction between computers and human language, enabling machines to understand and generate text."
        })

        # Словарь синонимов для улучшения поиска
        self.synonyms = {
            "agent": ["агент", "помощник", "ассистент", "бот"],
            "ai": ["artificial intelligence", "машинный интеллект", "ИИ"],
            "ml": ["machine learning", "машинное обучение", "ML"],
            "python": ["питон", "питон программирование"],
            "neural": ["нейронная", "нейросеть", "neural network"],
            "nlp": ["natural language processing", "обработка естественного языка", "NLP"]
        }

    def _expand_query_terms(self, query: str) -> set:
        """Расширяет запрос с учетом синонимов."""
        query_lower = query.lower()
        terms = set(query_lower.split())

        # Добавляем синонимы
        for key, synonyms in self.synonyms.items():
            if key in query_lower:
                terms.update(synonyms)
            for synonym in synonyms:
                if synonym in query_lower:
                    terms.add(key)
                    terms.update(synonyms)

        return terms

    def _calculate_relevance(self, doc: str, query_terms: set) -> float:
        """Вычисляет релевантность документа запросу."""
        doc_lower = doc.lower()
        matches = sum(1 for term in query_terms if term in doc_lower)
        return matches / len(query_terms) if query_terms else 0.0

    def retrieve_documents(self, query: str, top_k: int = 2) -> List[str]:
        """Улучшенный поиск документов с учетом синонимов и ранжированием."""
        query_terms = self._expand_query_terms(query)

        # Вычисляем релевантность для каждого документа
        scored_docs = []
        for key, doc in self.documents.items():
            relevance = self._calculate_relevance(doc, query_terms)
            if relevance > 0:
                scored_docs.append((relevance, doc))

        # Сортируем по релевантности (от большей к меньшей)
        scored_docs.sort(reverse=True, key=lambda x: x[0])

        # Возвращаем top_k наиболее релевантных документов
        return [doc for _, doc in scored_docs[:top_k]]

    def query(self, user_question: str, model: str = DEFAULT_MODEL, top_k: int = 2) -> str:
        """Query с настраиваемым top_k."""
        # Retrieve relevant documents
        relevant_docs = self.retrieve_documents(user_question, top_k=top_k)

        if not relevant_docs:
            return "Не найдено релевантных документов для ответа на ваш вопрос."

        # Create context-enhanced prompt
        context = "\n\n".join(relevant_docs)
        enhanced_prompt = f"""Context information:
{context}

Question: {user_question}

Please answer the question using the provided context information. If the context doesn't contain relevant information, say so clearly."""

        response = self.client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": enhanced_prompt}],
            temperature=0.1,
            max_tokens=1000
        )

        return response.choices[0].message.content

# Тестируем улучшенный RAG агент
print("=" * 70)
print("Тестирование улучшенного RAG Agent")
print("=" * 70)
enhanced_rag = EnhancedRAGAgent(client)
result = enhanced_rag.query("Что такое машинное обучение и как оно связано с нейронными сетями?", top_k=3)
print(f"\n Результат: {result}")



## Секция 3: LlamaIndex для RAG агентов

```
# Выбран кодовый формат
```



**LlamaIndex** специализируется на приложениях Retrieval-Augmented Generation (RAG), что делает его идеальным для агентов, работающих со знаниями.

In [ ]:
class SimpleWorkflowAgent:
    """A simple workflow agent inspired by LangGraph concepts."""

    def __init__(self, client, tools, function_map):
        self.client = client
        self.tools = tools
        self.function_map = function_map
        self.state = {} # workflow like sys basis

    def plan_workflow(self, user_request: str) -> List[str]:
        """Plan the workflow steps based on user request."""
        planning_prompt = f"""
        Given this user request: "{user_request}"

        Break it down into a sequence of steps. Each step should be one of:
        - "get_time" - get current time
        - "calculate" - perform calculation
        - "search" - search knowledge base
        - "summarize" - provide final summary
        - "analyze" - analyze calculations

        Return only the step names, one per line.
        """

        response = self.client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=[{"role": "user", "content": planning_prompt}],
            temperature=0.1
        )

        steps = [step.strip() for step in response.choices[0].message.content.split('\n') if step.strip()]
        return steps

    def execute_step(self, step: str, context: str) -> str:
        """Execute a single workflow step."""
        if step == "get_time":
            return get_current_time()
        elif step == "calculate":
            # Extract calculation from context
            calc_prompt = f"Extract the mathematical operation from: {context}. Return in format: operation,a,b (e.g., +,5,3)"
            print(calc_prompt)
            response = self.client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=[{"role": "user", "content": calc_prompt}],
                temperature=0.1
            )
            print(response)
            try:
                parts = response.choices[0].message.content.strip().split(',')
                if len(parts) == 3:
                    op, a, b = parts[0].strip(), float(parts[1].strip()), float(parts[2].strip())
                    return str(calculate(op, a, b))
            except:
                pass
            return "Could not extract calculation"
        elif step == "search":
            # Extract search query from context
            search_prompt = f"Extract the main topic to search from: {context}. Return only the topic name."
            response = self.client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=[{"role": "user", "content": search_prompt}],
                temperature=0.1
            )
            query = response.choices[0].message.content.strip()
            return search_knowledge_base(query)
        elif step == "summarize":
            summary_prompt = f"Summarize the following information for the user: {context}"
            response = self.client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=[{"role": "user", "content": summary_prompt}],
                temperature=0.1
            )
            return response.choices[0].message.content
        elif step == "analyze":
          analysis_prompt = f"Analyze the following result and give your comments {context} check if that right"
          response = self.client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=[{"role": "user", "content": analysis_prompt}],
                temperature=0.1
          )
          return response.choices[0].message.content
        else:
            return f"Unknown step: {step}"

    def run_workflow(self, user_request: str) -> str:
        """Run the complete workflow."""
        print(f"Planning workflow for: {user_request}")

        # Plan the workflow
        steps = self.plan_workflow(user_request)
        print(f"Planned steps: {steps}")

        # Execute each step
        results = []
        context = user_request

        for i, step in enumerate(steps):
            print(f"\nExecuting step {i+1}: {step}")

            if step == "calculate":
              result = self.execute_step("calculate", context)
              results.append(f"Step {i+1} ({step}): {result}")
              context += f" | {result}"
              calc_value = float(result)

              if calc_value > 50:
                analysis_result = self.execute_step("analyze", context)
                results.append(f"Step {i+1} ({step}): {result}")
                context += f" | {analysis_result}"

            else:
              result = self.execute_step(step, context)
              results.append(f"Step {i+1} ({step}): {result}")
              context += f" | {result}"
              print(f"Result: {result}")

        # Final summary
        final_summary = self.execute_step("summarize", " | ".join(results))
        return final_summary

# Create and test workflow agent
workflow_agent = SimpleWorkflowAgent(client, tools, function_map)
print("Workflow Agent created")

# Test Workflow Agent
print("\nTesting Workflow Agent:")
workflow_result = workflow_agent.run_workflow("What time is it and calculate 20 * 3?")
print(f"Workflow Agent Result: {workflow_result}")

Workflow Agent created

Testing Workflow Agent:
Planning workflow for: What time is it and calculate 20 * 3?
Planned steps: ['get_time', 'calculate', 'summarize']

Executing step 1: get_time
Result: 2025-11-13 17:31:15 UTC

Executing step 2: calculate
Extract the mathematical operation from: What time is it and calculate 20 * 3? | 2025-11-13 17:31:15 UTC. Return in format: operation,a,b (e.g., +,5,3)
ChatCompletion(id='gen-1763055075-x4OtW4llgBStgC05DVrI', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='*,20,3', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='stop')], created=1763055075, model='qwen/qwen-2.5-72b-instruct', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=6, prompt_tokens=67, total_tokens=73, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audi

---

## Задание 4: Создание собственного Workflow

**Цель** Научиться проектировать многошаговые workflow для решения конкретных задач.

**Что нужно сделать**
Создайте workflow агента для одной из следующих задач (или придумайте свою):
- Генерация краткого отчета на основе нескольких запросов
- Анализ данных с последующей визуализацией
- Поиск информации и создание резюме

**Требования:**
1. Определите минимум 3 шага для вашего workflow
2. Добавьте условное ветвление (например, если результат шага 1 > X, то делаем шаг A, иначе шаг B)
3. Реализуйте обработку ошибок для каждого шага

**Подсказки**
- Используйте `SimpleWorkflowAgent` как основу
- Добавьте новые типы шагов в `execute_step`
- Используйте `self.state` для хранения промежуточных результатов

In [ ]:
# Задание 4: Создайте свой workflow здесь

class CustomWorkflowAgent(SimpleWorkflowAgent):
    """Ваш кастомный workflow агент."""

    def plan_workflow(self, user_request: str) -> List[str]:
        """Планируйте workflow для вашей задачи."""
        # Определите шаги для вашего workflow
        # Например: ["step1", "step2", "step3"]
        pass

    def execute_step(self, step: str, context: str) -> str:
        """Выполните шаг с условным ветвлением."""
        # Реализуйте ваши шаги
        # Добавьте условную логику на основе context или self.state
        pass

    def run_workflow(self, user_request: str) -> str:
        """Запустите workflow с обработкой ошибок."""
        # Модифицируйте метод для добавления обработки ошибок
        pass

# Протестируйте ваш workflow
# custom_agent = CustomWorkflowAgent(client, tools, function_map)
# result = custom_agent.run_workflow("Ваш запрос здесь")
# print(result)

## Секция 5: Мульти-агентная система

Пример системы с несколькими специализированными агентами.

In [ ]:
class MultiAgentSystem:
    """A simple multi-agent system with specialized agents."""

    def __init__(self, client):
        self.client = client
        self.agents = {
            "calculator": self.create_calculator_agent(),
            "researcher": self.create_researcher_agent(),
            "coordinator": self.create_coordinator_agent()
        }

    def create_calculator_agent(self):
        """Agent specialized in mathematical operations."""
        def agent(query: str) -> str:
            calc_prompt = f"""You are a calculator agent. Your job is to perform mathematical calculations.

Query: {query}

If this contains a mathematical operation, perform it and return the result.
If not, respond with "Not a calculation task"."""

            response = self.client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=[{"role": "user", "content": calc_prompt}],
                temperature=0.1
            )
            return response.choices[0].message.content
        return agent

    def create_researcher_agent(self):
        """Agent specialized in information retrieval."""
        def agent(query: str) -> str:
            research_prompt = f"""You are a research agent. Your job is to find information.

Query: {query}

If this is asking for information, search your knowledge and provide a detailed answer.
If not, respond with "Not a research task"."""

            response = self.client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=[{"role": "user", "content": research_prompt}],
                temperature=0.1
            )
            return response.choices[0].message.content
        return agent

    def create_coordinator_agent(self):
        """Agent that coordinates other agents."""
        def agent(query: str, agent_results: Dict[str, str]) -> str:
            coord_prompt = f"""You are a coordinator agent. Your job is to combine results from other agents.

Original query: {query}

Agent results:
{json.dumps(agent_results, indent=2)}

Provide a comprehensive response that combines all relevant information."""

            response = self.client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=[{"role": "user", "content": coord_prompt}],
                temperature=0.1
            )
            return response.choices[0].message.content
        return agent

    def process_query(self, query: str) -> str:
        """Process a query using multiple agents."""
        print(f"Multi-agent system processing: {query}")

        # Get responses from specialized agents
        results = {}
        for agent_name, agent_func in self.agents.items():
            if agent_name != "coordinator":
                print(f"Consulting {agent_name} agent...")
                results[agent_name] = agent_func(query)

        # Coordinate the results
        print("Coordinating results...")
        final_result = self.agents["coordinator"](query, results)
        return final_result

# Create and test multi-agent system
multi_agent_system = MultiAgentSystem(client)
print("Multi-Agent System created")

# Test Multi-Agent System
print("\nTesting Multi-Agent System:")
multi_result = multi_agent_system.process_query("Calculate 50 / 2 and explain what AI agents are")
print(f"Multi-Agent Result: {multi_result}")


Multi-Agent System created

Testing Multi-Agent System:
Multi-agent system processing: Calculate 50 / 2 and explain what AI agents are
Consulting calculator agent...
Consulting researcher agent...
Coordinating results...
Multi-Agent Result: ### Calculation and Explanation of AI Agents

#### Calculation
The calculation of 50 / 2 is 25.

#### Explanation of AI Agents

**AI agents, or artificial intelligence agents, are software systems designed to perceive their environment and take actions that maximize their chances of achieving specific goals.** These agents can operate in a variety of environments, including digital, physical, and hybrid settings. Here’s a more detailed breakdown:

### Components of AI Agents
1. **Perception:**
   - AI agents use sensors or input mechanisms to gather data from their environment. This could be visual data from cameras, auditory data from microphones, or textual data from user inputs.

2. **Cognition:**
   - The agent processes the gathered data using 

## Секция 6: Сравнение фреймворков и лучшие практики (5 минут)

### Сравнение фреймворков:

**1. BASIC REACT AGENT**
- Сильные стороны: Простота понимания и реализации, хорош для обучения, прямой контроль над циклом рассуждений
- Слабые стороны: Требует ручной реализации, ограниченная обработка ошибок, нет встроенных оптимизаций
- Лучше для: Обучения, прототипирования, простых задач

**2. SMOL AGENTS**
- Сильные стороны: Минимальные зависимости, быстрое выполнение, легко кастомизировать
- Слабые стороны: Ограниченные встроенные функции, требует больше ручной работы
- Лучше для: Легковесных приложений, встроенных систем

**3. LLAMAINDEX (RAG AGENTS)**
- Сильные стороны: Отлично для задач с большим объемом знаний, встроенная обработка документов, оптимизированные механизмы поиска
- Слабые стороны: Сфокусирован в основном на RAG use cases, может быть избыточным для простых задач
- Лучше для: Вопрос-ответ, анализ документов, базы знаний

**4. LANGGRAPH (WORKFLOW AGENTS)**
- Сильные стороны: Управление сложными workflow, сохранение состояния, условная маршрутизация
- Слабые стороны: Более крутая кривая обучения, более сложная настройка
- Лучше для: Сложных многошаговых процессов, корпоративных приложений

**5. MULTI-AGENT SYSTEMS**
- Сильные стороны: Специализированная экспертиза, параллельная обработка, масштабируемая архитектура
- Слабые стороны: Сложность координации, более высокое использование ресурсов
- Лучше для: Сложных проблем, требующих разных областей экспертизы

### Резюме семинара:

**Что мы изучили:**
1. Основные концепции AI агентов и паттерн ReAct
2. Базовая реализация агента с вызовом инструментов
3. Smol Agents для легковесных приложений
4. LlamaIndex для RAG-агентов на основе знаний
5. Управление workflow в стиле LangGraph
6. Координация мульти-агентных систем
7. Сравнение фреймворков и критерии выбора
8. Соображения по развертыванию в продакшене

**Ключевые выводы:**
- Агенты расширяют LLM возможностями рассуждения и действия
- Разные фреймворки служат разным use cases
- Паттерн ReAct является фундаментальным для большинства архитектур агентов
- Развертывание в продакшене требует тщательного рассмотрения ошибок, затрат и безопасности